# RescueVision Edge — Experiment Report

**Research Question:** *How effective is 8-bit post-training quantization of YOLO-based
victim detection on ESP32-S3 for disaster SAR scenarios?*

This notebook documents the full experiment pipeline: dataset preparation, baseline training,
quantization, edge deployment, and comparative analysis across PC, Raspberry Pi, and ESP32-S3.

In [ ]:
import sys, json, yaml, time, warnings
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid')

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
import utils

print('Project root:', PROJECT_ROOT)

## 1. Configuration

In [ ]:
cfg = utils.load_config()
print(yaml.dump(cfg, default_flow_style=False))

## 2. Dataset

COCO person subset: filtering images containing 'person' class, splitting 80/15/5.

In [ ]:
dataset_dir = utils.DATASET_DIR
for split in ['train', 'val', 'test']:
    img_dir = dataset_dir / 'images' / split
    lbl_dir = dataset_dir / 'labels' / split
    n_imgs = len(list(img_dir.glob('*'))) if img_dir.exists() else 0
    n_lbls = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
    print(f'{split}: {n_imgs} images, {n_lbls} labels')

## 3. Baseline FP32 Model

YOLOv8n pretrained on COCO, fine-tuned on person detection.

In [ ]:
baseline_results = utils.load_results(utils.BASELINE_DIR, 'fp32_baseline') \
    if (utils.BASELINE_DIR / 'results_fp32_baseline.json').exists() else {}
print(json.dumps(baseline_results, indent=2)[:500])

## 4. Quantization Results

Post-training quantization: FP32 → INT8 via TFLite.

In [ ]:
quant_results = utils.load_results(utils.QUANTIZED_DIR, 'quantization_comparison') \
    if (utils.QUANTIZED_DIR / 'results_quantization_comparison.json').exists() else {}

if quant_results:
    for variant, data in quant_results.get('variants', {}).items():
        print(f"{variant}:")
        print(f"  Size: {data.get('size_mb', 'N/A')} MB")
        print(f"  mAP@0.5: {data.get('mAP@0.5', 'N/A')}")
        print(f"  Latency: {data.get('mean_latency_ms', 'N/A')} ms")
        print(f"  FPS: {data.get('fps', 'N/A')}\n")

### Accuracy-Size Trade-off

In [ ]:
if quant_results:
    variants = quant_results.get('variants', {})
    names = list(variants.keys())
    sizes = [v.get('size_mb', 0) for v in variants.values()]
    maps = [v.get('mAP@0.5', 0) for v in variants.values()]
    lats = [v.get('mean_latency_ms', 0) for v in variants.values()]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].bar(names, sizes, color=['blue', 'green', 'red'])
    axes[0].set_title('Model Size (MB)')
    axes[0].set_xticklabels(names, rotation=15)

    axes[1].bar(names, maps, color=['blue', 'green', 'red'])
    axes[1].set_title('mAP@0.5')
    axes[1].set_xticklabels(names, rotation=15)

    axes[2].bar(names, lats, color=['blue', 'green', 'red'])
    axes[2].set_title('Latency (ms)')
    axes[2].set_xticklabels(names, rotation=15)

    plt.tight_layout()
    plt.savefig(utils.PROJECT_ROOT / 'notebooks' / 'quantization_tradeoff.png', dpi=150)
    plt.show()

## 5. Resolution Benchmark

INT8 model tested at 192×192, 256×256, 320×320 input resolutions.

In [ ]:
eval_results = utils.load_results(utils.PROJECT_ROOT / 'evaluation_results', 'full_evaluation') \
    if (utils.PROJECT_ROOT / 'evaluation_results' / 'results_full_evaluation.json').exists() else {}

if eval_results:
    int8 = eval_results.get('tflite_int8', {}).get('resolution_benchmarks', {})
    fp32 = eval_results.get('tflite_fp32', {}).get('resolution_benchmarks', {})

    print(f"{'Resolution':<12} {'INT8 FPS':<12} {'FP32 FPS':<12} {'Speedup':<10}")
    print('-' * 46)
    for res in sorted(int8.keys()):
        i_fps = int8[res].get('fps', 'N/A')
        f_fps = fp32.get(res, {}).get('fps', 'N/A')
        speedup = f_fps / i_fps if isinstance(f_fps, (int,float)) and isinstance(i_fps, (int,float)) and i_fps > 0 else 'N/A'
        speedup_str = f'{speedup:.2f}x' if isinstance(speedup, float) else str(speedup)
        print(f"{res:<12} {str(i_fps):<12} {str(f_fps):<12} {speedup_str:<10}")

## 6. ESP32-S3 Feasibility

In [ ]:
feas = eval_results.get('esp32s3_feasibility', {})
if feas:
    print(f"Flash required: {feas.get('flash_required_mb', '?')} MB")
    print(f"RAM required: {feas.get('ram_required_kb', '?')} KB")
    print(f"ESP32-S3 flash: {feas.get('esp32s3_available_flash_mb', '?')} MB")
    print(f"ESP32-S3 PSRAM: {feas.get('esp32s3_available_psram_mb', '?')} MB")
    print(f"Feasible: {'✅ YES' if feas.get('feasible') else '❌ NO'}")

## 7. Summary Table

| Variant | Size (MB) | mAP@0.5 | FPS @192 | FPS @320 |
|---------|-----------|---------|----------|----------|

In [ ]:
if eval_results:
    print(f"| {'Variant':<20} | {'Size':<8} | {'mAP@0.5':<8} | {'FPS@192':<8} | {'FPS@320':<8} |")
    print(f"| {'-'*20} | {'-'*8} | {'-'*8} | {'-'*8} | {'-'*8} |")
    for var_name in ['tflite_fp32', 'tflite_int8']:
        v = eval_results.get(var_name, {})
        size = v.get('file_size_mb', 'N/A')
        map_val = eval_results.get('quantization_comparison', {}).get('variants', {}).get(var_name.replace('tflite_', 'TFLite ').replace('_', ' ').title(), {}).get('mAP@0.5', 'N/A')
        fps192 = v.get('resolution_benchmarks', {}).get('192x192', {}).get('fps', 'N/A')
        fps320 = v.get('resolution_benchmarks', {}).get('320x320', {}).get('fps', 'N/A')
        label = var_name.replace('tflite_', '').upper()
        print(f"| {label:<20} | {str(size):<8} | {str(map_val):<8} | {str(fps192):<8} | {str(fps320):<8} |")

print('\nExperiment complete.')